<a href="https://colab.research.google.com/github/Salimajeeba/ai-engineer-enterprise-knowledge-base-rag/blob/main/Enterprise_Knowledge_Base_Semantic_Search_and_RAG_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sentence-transformers chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 51.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 61.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.1 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0.post0
    Uninstalling python-dateutil-2.9.0.post0:
      Successfully uninstalled python-dateutil-2.9.0.post0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: oauthlib
    Found existing installation: oauthlib 3.3.1
    Uninstalling oauthlib-3.3.1:
      Successfully uninstalled oauthlib-3.3.1
  Attempting uninstall: opentelemetry-semantic-conven

### Verify Installations
Let's verify that both `sentence_transformers` and `chromadb` can be imported successfully.

### Restart Runtime
Run the cell below to programmatically restart your Colab runtime. This will clear the active memory cache and allow PyTorch to load correctly. After the restart, run the verification cell again.

In [ ]:
import os
# Programmatically restart the Colab runtime session
os.kill(os.getpid(), 9)

In [3]:
import os
from typing import List, Dict, Any
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
from chromadb.config import Settings

# 1. Initialize Embedding Model & Reranker
print("[INFO] Loading Embedding and Reranker Models...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # 384-dim vector model
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# 2. Document Ingestion & Chunking Function
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

# Sample Enterprise Documents
raw_documents = [
    {
        "id": "doc_1",
        "title": "Database Storage Policies",
        "category": "Vector Storage and Indexing",
        "content": "Vector databases like ChromaDB and Pinecone use HNSW (Hierarchical Navigable Small World) graphs and IVF (Inverted File) indexing to achieve ultra-fast similarity searches across high-dimensional vectors."
    },
    {
        "id": "doc_2",
        "title": "Embedding Mathematics",
        "category": "Embedding Fundamentals",
        "content": "Text embeddings transform semantic textual information into dense numerical vectors. Similarity between vectors is calculated using metrics like Cosine Similarity, Dot Product, and Euclidean Distance."
    },
    {
        "id": "doc_3",
        "title": "Document Chunking Strategies",
        "category": "Document Processing",
        "content": "Chunking is the process of breaking long documents into smaller segments. Recursive character text splitters maintain document structure, while semantic chunking groups sentences by contextual similarity."
    },
    {
        "id": "doc_4",
        "title": "Advanced Retrieval & Reranking",
        "category": "Search and Retrieval",
        "content": "Hybrid search combines dense vector retrieval with sparse keyword search (BM25). Reranking models score initial vector matches using cross-encoders to dramatically increase top-K search precision."
    }
]

# 3. Setup ChromaDB Vector Store
print("[INFO] Initializing ChromaDB Vector Store...")
chroma_client = chromadb.Client(Settings(allow_reset=True))
collection = chroma_client.create_collection(name="enterprise_knowledge_base")

# 4. Processing & Indexing Data
documents_to_add = []
metadatas_to_add = []
ids_to_add = []
embeddings_to_add = []

for doc_obj in raw_documents:
    chunks = chunk_text(doc_obj["content"])
    for idx, chunk in enumerate(chunks):
        chunk_id = f"{doc_obj['id']}_chunk_{idx}"
        vector = embedding_model.encode(chunk).tolist()

        documents_to_add.append(chunk)
        metadatas_to_add.append({"title": doc_obj["title"], "category": doc_obj["category"]})
        ids_to_add.append(chunk_id)
        embeddings_to_add.append(vector)

collection.add(
    documents=documents_to_add,
    embeddings=embeddings_to_add,
    metadatas=metadatas_to_add,
    ids=ids_to_add
)
print(f"[SUCCESS] Indexed {len(ids_to_add)} vector chunks into ChromaDB.")

# 5. Semantic Search & Reranking Pipeline
def pipeline_semantic_search(query: str, category_filter: str = None, top_k: int = 3):
    print(f"\n--- Executing Query: '{query}' ---")
    query_vector = embedding_model.encode(query).tolist()

    where_clause = {"category": category_filter} if category_filter else None

    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k,
        where=where_clause
    )

    retrieved_chunks = results['documents'][0]
    retrieved_meta = results['metadatas'][0]

    # Reranking Top Matches
    pairs = [[query, doc] for doc in retrieved_chunks]
    rerank_scores = reranker_model.predict(pairs)

    ranked_results = sorted(zip(retrieved_chunks, retrieved_meta, rerank_scores), key=lambda x: x[2], reverse=True)

    for idx, (doc, meta, score) in enumerate(ranked_results):
        print(f"Rank {idx+1} [Rerank Score: {score:.4f}] | Category: {meta['category']} | Title: {meta['title']}")
        print(f"   Snippet: {doc}\n")

# Run Examples
pipeline_semantic_search(query="How does vector indexing work?", top_k=2)
pipeline_semantic_search(query="Tell me about similarity metrics and vectors", category_filter="Embedding Fundamentals", top_k=1)

[INFO] Loading Embedding and Reranker Models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[INFO] Initializing ChromaDB Vector Store...
[SUCCESS] Indexed/Updated 4 vector chunks in ChromaDB.

--- Executing Query: 'How does vector indexing work?' ---
Rank 1 [Rerank Score: 2.1570] | Category: Vector Storage and Indexing | Title: Database Storage Policies
   Snippet: Vector databases like ChromaDB and Pinecone use HNSW (Hierarchical Navigable Small World) graphs and IVF (Inverted File) indexing to achieve ultra-fast similarity searches across high-dimensional vectors.

Rank 2 [Rerank Score: -5.3959] | Category: Search and Retrieval | Title: Advanced Retrieval & Reranking
   Snippet: Hybrid search combines dense vector retrieval with sparse keyword search (BM25). Reranking models score initial vector matches using cross-encoders to dramatically increase top-K search precision.


--- Executing Query: 'Tell me about similarity metrics and vectors' ---
Rank 1 [Rerank Score: 3.9829] | Category: Embedding Fundamentals | Title: Embedding Mathematics
   Snippet: Text embeddings transfo